In [1]:
import cv2
import dlib
import pyttsx3
from scipy.spatial import distance
import os
import time

engine = pyttsx3.init()

cap = cv2.VideoCapture(0)
face_detector = dlib.get_frontal_face_detector()

dat_file = "shape_predictor_68_face_landmarks.dat"
if not os.path.exists(dat_file):
    raise FileNotFoundError(f"{dat_file} not found in current folder.")

dlib_facelandmark = dlib.shape_predictor(dat_file)

# EAR (Eye Aspect Ratio)
def detect_eye(eye_points):
    A = distance.euclidean(eye_points[1], eye_points[5])
    B = distance.euclidean(eye_points[2], eye_points[4])
    C = distance.euclidean(eye_points[0], eye_points[3])
    return (A + B) / (2.0 * C)

# MAR (Mouth Aspect Ratio)
def detect_mouth(mouth_points):
    A = distance.euclidean(mouth_points[2], mouth_points[10])  # 51–59
    B = distance.euclidean(mouth_points[4], mouth_points[8])   # 53–57
    C = distance.euclidean(mouth_points[0], mouth_points[6])   # 49–55
    return (A + B) / (2.0 * C)

# Thresholds
EAR_THRESHOLD = 0.24
EAR_ALERT_DELAY = 5
eye_closed_start = None
alert_triggered = False

MAR_THRESHOLD = 0.6
MAR_ALERT_DELAY = 3
mouth_open_start = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector(gray)

    for face in faces:
        landmarks = dlib_facelandmark(gray, face)

        # Eyes
        left_eye, right_eye = [], []
        for n in range(42, 48):  # Re
            x, y = landmarks.part(n).x, landmarks.part(n).y
            right_eye.append((x, y))
            next_point = 42 if n == 47 else n + 1
            cv2.line(frame, (x, y), (landmarks.part(next_point).x, landmarks.part(next_point).y), (0, 255, 0), 1)

        for n in range(36, 42):  # Le
            x, y = landmarks.part(n).x, landmarks.part(n).y
            left_eye.append((x, y))
            next_point = 36 if n == 41 else n + 1
            cv2.line(frame, (x, y), (landmarks.part(next_point).x, landmarks.part(next_point).y), (0, 255, 0), 1)

        left_EAR = detect_eye(left_eye)
        right_EAR = detect_eye(right_eye)
        EAR = (left_EAR + right_EAR) / 2.0

        # M
        mouth = []
        for n in range(48, 68):
            x, y = landmarks.part(n).x, landmarks.part(n).y
            mouth.append((x, y))
            next_point = 48 if n == 67 else n + 1
            cv2.line(frame, (x, y), (landmarks.part(next_point).x, landmarks.part(next_point).y), (255, 255, 0), 1)

        MAR = detect_mouth(mouth)

        # Eye logic
        if EAR < EAR_THRESHOLD:
            if eye_closed_start is None:
                eye_closed_start = time.time()
            elapsed_eye = time.time() - eye_closed_start
            if elapsed_eye >= EAR_ALERT_DELAY:
                cv2.putText(frame, "DROWSINESS DETECTED", (50, 100),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                cv2.putText(frame, "WAKE UP!", (50, 150),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

                if not alert_triggered:
                    engine.say("Alert! Wake up Buddy")
                    engine.say("Wake up to reality")
                    engine.runAndWait()
                    alert_triggered = True
        else:
            eye_closed_start = None
            alert_triggered = False

        # Mouth logic
        if MAR > MAR_THRESHOLD:
            if mouth_open_start is None:
                mouth_open_start = time.time()
            elapsed_mouth = time.time() - mouth_open_start
            if elapsed_mouth >= MAR_ALERT_DELAY:
                cv2.putText(frame, "YAWN DETECTED", (50, 200),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                cv2.putText(frame, "TAKE A BREAK!", (50, 250),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        else:
            mouth_open_start = None

        # Display values
        cv2.putText(frame, f"EAR: {EAR:.2f}", (450, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"MAR: {MAR:.2f}", (450, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    cv2.imshow("Drowsiness + Yawn Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('s'):
        break

cap.release()
cv2.destroyAllWindows()